In [1]:
from pathlib import Path
from urllib.parse import urljoin, urlparse
from html import unescape
import re

import pandas as pd
import requests

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Raw folder exists:", RAW_DATA_DIR.is_dir())
print("Processed folder exists:", PROCESSED_DATA_DIR.is_dir())

Project root: /Users/dafyddjones/projects/finance-kpi-reporting-pipeline
Raw folder exists: True
Processed folder exists: True


In [2]:
monthly_pages = {
    "2025-10": "october-2025",
    "2025-11": "november-2025",
    "2025-12": "december-2025",
    "2026-01": "january-2026",
    "2026-02": "february-2026",
    "2026-03": "march-2026",
}

base_url = (
    "https://www.gov.uk/government/publications/"
    "dft-spending-over-25000-"
)

publication_urls = {
    month: base_url + page_name
    for month, page_name in monthly_pages.items()
}

for month, url in publication_urls.items():
    print(month, "→", url)

2025-10 → https://www.gov.uk/government/publications/dft-spending-over-25000-october-2025
2025-11 → https://www.gov.uk/government/publications/dft-spending-over-25000-november-2025
2025-12 → https://www.gov.uk/government/publications/dft-spending-over-25000-december-2025
2026-01 → https://www.gov.uk/government/publications/dft-spending-over-25000-january-2026
2026-02 → https://www.gov.uk/government/publications/dft-spending-over-25000-february-2026
2026-03 → https://www.gov.uk/government/publications/dft-spending-over-25000-march-2026


In [3]:
def download_dft_csv(month, publication_url):
    file_path = RAW_DATA_DIR / (
        f"dft_spending_{month.replace('-', '_')}.csv"
    )

    # Reuse an existing non-empty file.
    if file_path.exists() and file_path.stat().st_size > 0:
        print(f"{month}: using existing file")
        return file_path

    # Retrieve the publication page.
    page = requests.get(publication_url, timeout=30)
    page.raise_for_status()

    links = re.findall(
        r'href="([^"]+\.csv(?:\?[^"]*)?)"',
        page.text,
        flags=re.IGNORECASE
    )

    urls = {
        urljoin(publication_url, unescape(link))
        for link in links
    }

    download_urls = [
        url for url in urls
        if urlparse(url).hostname
        == "assets.publishing.service.gov.uk"
    ]

    if len(download_urls) != 1:
        raise ValueError(
            f"{month}: expected one CSV download link; "
            f"found {len(download_urls)}"
        )

    # Download and save the original bytes.
    download = requests.get(download_urls[0], timeout=30)
    download.raise_for_status()

    if not download.content:
        raise ValueError(f"{month}: downloaded file is empty")

    if "text/html" in download.headers.get(
        "Content-Type", ""
    ).lower():
        raise ValueError(f"{month}: received HTML instead of CSV")

    file_path.write_bytes(download.content)

    print(f"{month}: downloaded {len(download.content):,} bytes")
    return file_path

In [4]:
monthly_files = {}

for month, publication_url in publication_urls.items():
    monthly_files[month] = download_dft_csv(
        month,
        publication_url
    )

print("\nFiles ready:", len(monthly_files))

2025-10: downloaded 507,446 bytes
2025-11: downloaded 458,640 bytes
2025-12: downloaded 491,912 bytes
2026-01: using existing file
2026-02: downloaded 473,657 bytes
2026-03: downloaded 668,070 bytes

Files ready: 6


In [5]:
monthly_tables = {}
file_checks = []

for month, path in monthly_files.items():
    try:
        table = pd.read_csv(
            path, encoding="utf-8-sig", dtype=str
        )
        encoding = "utf-8-sig"
    except UnicodeDecodeError:
        table = pd.read_csv(
            path, encoding="cp1252", dtype=str
        )
        encoding = "cp1252"

    table.columns = (
        table.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    table = table.rename(columns={"£": "amount_gbp"})

    monthly_tables[month] = table

    file_checks.append({
        "month": month,
        "rows": len(table),
        "columns": len(table.columns),
        "encoding": encoding,
        "matches_january_columns": table.columns.equals(
            monthly_tables["2026-01"].columns
        ) if "2026-01" in monthly_tables else None
    })

# Compare every month with January, including earlier months.
january_columns = monthly_tables["2026-01"].columns

for check in file_checks:
    check["matches_january_columns"] = (
        monthly_tables[check["month"]].columns.equals(
            january_columns
        )
    )

display(pd.DataFrame(file_checks))

,month,rows,columns,encoding,matches_january_columns
0,2025-10,2715,10,cp1252,True
1,2025-11,2446,10,cp1252,True
2,2025-12,2626,10,cp1252,True
3,2026-01,2426,10,cp1252,True
4,2026-02,2511,10,cp1252,True
5,2026-03,3593,10,cp1252,True


In [6]:
tables_to_combine = []

for month, table in monthly_tables.items():
    monthly_table = table.copy()

    monthly_table["source_month"] = month
    monthly_table["source_file"] = monthly_files[month].name
    monthly_table["source_row"] = range(1, len(monthly_table) + 1)

    tables_to_combine.append(monthly_table)

df_all = pd.concat(tables_to_combine, ignore_index=True)

print("Combined rows:", len(df_all))
print("Combined columns:", len(df_all.columns))

display(df_all.groupby("source_month").size().rename("records"))

Combined rows: 16317
Combined columns: 13


source_month
2025-10    2715
2025-11    2446
2025-12    2626
2026-01    2426
2026-02    2511
2026-03    3593
Name: records, dtype: int64

In [7]:
df_all["entity"] = df_all["entity"].str.strip()

df_dft = df_all.loc[
    df_all["entity"] == "Department for Transport"
].copy()

df_dft["payment_date"] = pd.to_datetime(
    df_dft["date"].str.strip(),
    format="%d/%m/%Y",
    errors="coerce"
)

amount_text = (
    df_dft["amount_gbp"]
    .str.strip()
    .str.replace("£", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace(r"^\((.*)\)$", r"-\1", regex=True)
)

df_dft["payment_amount_gbp"] = pd.to_numeric(
    amount_text,
    errors="coerce"
)

print("Central DfT records:", len(df_dft))
print("Invalid or missing dates:", df_dft["payment_date"].isna().sum())
print(
    "Invalid or missing amounts:",
    df_dft["payment_amount_gbp"].isna().sum()
)

display(
    df_dft.groupby("source_month").size().rename("central_dft_records")
)

Central DfT records: 3845
Invalid or missing dates: 0
Invalid or missing amounts: 0


source_month
2025-10    711
2025-11    531
2025-12    543
2026-01    659
2026-02    606
2026-03    795
Name: central_dft_records, dtype: int64

In [8]:
df_dft["payment_month"] = (
    df_dft["payment_date"].dt.strftime("%Y-%m")
)

month_mismatch = (
    df_dft["payment_month"] != df_dft["source_month"]
)

print("Records outside their source month:", month_mismatch.sum())

date_check = df_dft.groupby("source_month").agg(
    records=("payment_date", "size"),
    earliest_date=("payment_date", "min"),
    latest_date=("payment_date", "max")
)

display(date_check)

if month_mismatch.any():
    display(
        df_dft.loc[
            month_mismatch,
            ["source_month", "payment_date", "supplier",
             "payment_amount_gbp"]
        ].head(10)
    )

Records outside their source month: 0


,records,earliest_date,latest_date
source_month,,,
2025-10,711,2025-10-01,2025-10-31
2025-11,531,2025-11-03,2025-11-28
2025-12,543,2025-12-01,2025-12-31
2026-01,659,2026-01-02,2026-01-30
2026-02,606,2026-02-02,2026-02-27
2026-03,795,2026-03-02,2026-03-31


In [9]:
original_columns = monthly_tables["2026-01"].columns.tolist()

df_dft["identical_record_flag"] = df_dft.duplicated(
    subset=original_columns,
    keep=False
)

quality_summary = df_dft.groupby("source_month").agg(
    records=("payment_amount_gbp", "size"),
    identical_record_rows=("identical_record_flag", "sum"),
    missing_suppliers=("supplier", lambda s: s.isna().sum()),
    unavailable_descriptions=(
        "item_text",
        lambda s: s.str.strip().eq("#").sum()
    ),
    negative_amounts=(
        "payment_amount_gbp",
        lambda s: (s < 0).sum()
    ),
    zero_amounts=(
        "payment_amount_gbp",
        lambda s: (s == 0).sum()
    ),
    positive_amounts_up_to_25000=(
        "payment_amount_gbp",
        lambda s: ((s > 0) & (s <= 25_000)).sum()
    )
)

display(quality_summary)

,records,identical_record_rows,missing_suppliers,unavailable_descriptions,negative_amounts,zero_amounts,positive_amounts_up_to_25000
source_month,,,,,,,
2025-10,711,4,0,173,0,0,0
2025-11,531,6,0,141,0,0,0
2025-12,543,13,0,180,0,0,0
2026-01,659,8,0,154,0,0,0
2026-02,606,10,0,174,0,0,0
2026-03,795,8,0,271,0,0,0


In [10]:
monthly_summary = (
    df_dft.groupby("payment_month")
    .agg(
        payment_records=("payment_amount_gbp", "size"),
        distinct_supplier_names=("supplier", "nunique"),
        published_payments_gbp=("payment_amount_gbp", "sum"),
        median_payment_gbp=("payment_amount_gbp", "median"),
        largest_payment_gbp=("payment_amount_gbp", "max")
    )
    .sort_index()
)

monthly_summary["month_on_month_pct"] = (
    monthly_summary["published_payments_gbp"]
    .pct_change(fill_method=None) * 100
)

display(
    monthly_summary.style.format({
        "payment_records": "{:,.0f}",
        "distinct_supplier_names": "{:,.0f}",
        "published_payments_gbp": "£{:,.2f}",
        "median_payment_gbp": "£{:,.2f}",
        "largest_payment_gbp": "£{:,.2f}",
        "month_on_month_pct": "{:+.2f}%"
    }, na_rep="—")
)

,payment_records,distinct_supplier_names,published_payments_gbp,median_payment_gbp,largest_payment_gbp,month_on_month_pct
payment_month,,,,,,
2025-10,711,352,"£3,123,556,273.58","£182,006.55","£549,580,000.00",—
2025-11,531,275,"£2,366,997,852.94","£116,096.00","£534,722,000.00",-24.22%
2025-12,543,283,"£2,900,228,687.01","£132,220.40","£470,000,000.00",+22.53%
2026-01,659,336,"£2,391,570,289.87","£165,540.00","£423,374,000.00",-17.54%
2026-02,606,298,"£2,624,735,301.42","£106,732.71","£559,363,674.00",+9.75%
2026-03,795,424,"£3,202,766,377.35","£92,000.00","£487,671,000.00",+22.02%


In [11]:
combined_path = (
    PROCESSED_DATA_DIR
    / "dft_central_spending_2025_10_to_2026_03_cleaned.csv"
)

summary_path = (
    PROCESSED_DATA_DIR
    / "dft_monthly_spending_summary.csv"
)

df_dft.to_csv(combined_path, index=False)

monthly_summary.to_csv(
    summary_path,
    index=True,
    index_label="payment_month"
)

print("Cleaned records saved:", len(df_dft))
print("Monthly summary rows saved:", len(monthly_summary))
print("Combined data:", combined_path)
print("Monthly summary:", summary_path)

Cleaned records saved: 3845
Monthly summary rows saved: 6
Combined data: /Users/dafyddjones/projects/finance-kpi-reporting-pipeline/data/processed/dft_central_spending_2025_10_to_2026_03_cleaned.csv
Monthly summary: /Users/dafyddjones/projects/finance-kpi-reporting-pipeline/data/processed/dft_monthly_spending_summary.csv
